<a href="https://colab.research.google.com/github/aalugudedeepyasiri2005-cmd/PDF-RAG/blob/main/PDF_RAG_Project_With_Frontend.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# PDF-Based Retrieval-Augmented Generation (RAG) with Front-End

**Google Colab implementation**

This notebook follows the same pipeline as the classroom screenshots:
PDF Loader → Text Splitting → Hugging Face Embeddings → FAISS → Retrieval → Gemini LLM → Gradio Front-End.

**How to use:** Run the cells from top to bottom. Upload a PDF, enter your Gemini API key when prompted, process the document, and ask questions.

## 1. Install required libraries

In [22]:
from google.colab import drive

drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [23]:
!pip install -q langchain langchain-community langchain-core langchain-text-splitters langchain-huggingface langchain-google-genai sentence-transformers faiss-cpu pymupdf gradio

In [24]:
import os
import gradio as gr

from google.colab import drive

from langchain_community.document_loaders import PyMuPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate

print("Libraries imported successfully!")

Libraries imported successfully!


In [25]:
pdf_path = "/content/drive/MyDrive/Colab Notebooks/PE Project/TCS-annual-report-2024-2025.pdf"

print("PDF Path:", pdf_path)

print("File exists:", os.path.exists(pdf_path))

PDF Path: /content/drive/MyDrive/Colab Notebooks/PE Project/TCS-annual-report-2024-2025.pdf
File exists: True


In [26]:
loader = PyMuPDFLoader(pdf_path)

docs = loader.load()

print("Total Pages:", len(docs))

print(docs[0].page_content[:1000])

Total Pages: 336
2024-25
Integrated Annual Report


In [27]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=400,
    chunk_overlap=128
)

chunks = splitter.split_documents(docs)

print("Total Chunks:", len(chunks))

print(chunks[0].page_content)

Total Chunks: 3920
2024-25
Integrated Annual Report


In [28]:
embeddings = HuggingFaceEmbeddings(
    model_name="sentence-transformers/all-MiniLM-L6-v2"
)

print("Embeddings created successfully!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Embeddings created successfully!


In [29]:
vector_store = FAISS.from_documents(
    chunks,
    embeddings
)

print("FAISS database created successfully!")

FAISS database created successfully!


In [30]:
import re

# Extract PDF metadata
metadata = docs[0].metadata

# Get document title
document_title = metadata.get("title", "")

# If title is missing, use the first non-empty line
if not document_title:
    for doc in docs:
        lines = doc.page_content.strip().split("\n")
        for line in lines:
            if line.strip():
                document_title = line.strip()
                break
        if document_title:
            break

# Calculate total word count
full_text = " ".join(doc.page_content for doc in docs)

word_count = len(re.findall(r"\b\w+\b", full_text))

print("Document Title:", document_title)
print("Total Words:", word_count)

Document Title: TCS Integrated Annual Report 2024-25 comp
Total Words: 184822


In [31]:
query = "What is the main topic of this document?"

results = vector_store.similarity_search(query, k=4)

for result in results:
    print(result.page_content)
    print("--------------------")

Su s tain ab ility c o n text an d valu e c hain
Stak eh older engagement, 
data collection and analy sis
Identification of U niv erse of 
Relevant material ESG topics
Stak eh older Identification
M
aterial 
ESG Topics 
for T C S
Integrated Annual Report 2024-25
Su s tain ab ility Dis c lo s u res
319
1 GRI 3-1
--------------------
of material ESG topics, and the approach for assessing material 
issues involved is:
Stak eh older interactions result in th e identification 
of a b road funnel of issues important to each of th e 
constituencies. D iscussions w ith internal and ex ternal 
stak eh olders are ev aluated, to prioritiz e and arriv e at a list 
of material topics w ith significant economic, env ironmental,
--------------------
India in terms of Section 1ϰ3(11) of the Act, we give in 
the ͞ Annexure A͟ a statement on the matters specified in 
paragraphs 3 and ϰ of the Order, to the extent applicable.
2.
A.
As required by Section 1ϰ3(3) of the Act, we report that:
 
 
a.
We have 

In [32]:
from getpass import getpass

os.environ["GOOGLE_API_KEY"] = getpass("Enter Gemini API Key: ")

Enter Gemini API Key: ··········


In [33]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.8-flash",
    temperature=0
)

print("Gemini initialized successfully with gemini-3.8-flash!")

Gemini initialized successfully with gemini-3.8-flash!


In [34]:
retriever = vector_store.as_retriever(
    search_kwargs={"k": 4}
)

def rag_answer(question):

    q = question.lower()

    # Handle document title questions
    if "title" in q:
        return f"The title of the document is: {document_title}"

    # Handle word count questions
    if "word" in q and ("many" in q or "count" in q):
        return f"The document contains approximately {word_count} words."

    # Normal RAG retrieval
    retrieved_docs = retriever.invoke(question)

    if not retrieved_docs:
        return "Information not found in the PDF."

    context = "\n\n".join(
        doc.page_content for doc in retrieved_docs
    )

    prompt = f"""
    Answer using only the following PDF context.

    Context:
    {context}

    Question:
    {question}

    If the information is unavailable, say so.
    """

    response = llm.invoke(prompt)

    return response.content

In [35]:
question = "What is the main topic of this document?"

answer = rag_answer(question)

print(answer)

[{'type': 'text', 'text': 'Based on the provided context, an explicit "main topic" is not directly defined. However, the excerpts are from the **TCS Integrated Annual Report 2024-25**, primarily focusing on **Sustainability Disclosures**, **Stakeholder Engagement**, and the **identification and assessment of Material ESG Topics** (along with an excerpt from an auditor\'s report under Section 143 of the Act). \n\nIf you are looking for a specifically stated main topic for the entire document, that information is unavailable.', 'extras': {'signature': 'EqkWCqYWAWkUfROyQxfr/mvaCCgXZSOFQLOreOJRKlw5x7xyw7kHjQ0xqrV171y0M022KvvlNxAt0aU6ZROCmkzYWS1CqF+ff6NGonYPAfSC8tmlPkc/aCywwp689pGzyBXnJDdna32g27O/CTakeff1S0BFAMwsQZzVPoC+zDCh7xx/c0EvXaDtpovYX3zDD2M1AWJMutqYAD+o4Expo2eC6LkCjB4oivb9EFenqGctL4d5GRNcHkFIkXfx8/AV+uBgJHLcEiTb/OQ/MYJNbktH4qeou4BljIh5cyps1Fcbm7lildN4yOZpNl+2+aaS+uiExmY76PDdmwpavnNofvlloK5+fCVDU/beqgyLO6hxfxG30kr5Npb7fbyiacLIUOAqo4ioj6jGZ+o1Xbm/3OKx8bEAp6VWyXFloLdBjedQ6pW+lrGFzsGdadj

In [36]:
def chat(message, history):

    answer = rag_answer(message)

    return answer


demo = gr.ChatInterface(
    fn=chat,
    title="📚 PDF RAG Assistant",
    description="Upload your PDF through Google Drive and ask questions about its content."
)

demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://f5b0279171fc28133a.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
